### a. Definirea modelului

Starile ascunse sunt dificultatile testelor, in ordinea:

$$
[D,M,U]
$$

unde $D$ = dificil, $M$ = mediu si $U$ = usor.

Observatiile sunt notele, in ordinea:

$$
[FB,B,S,NS]
$$

Distributia initiala este uniforma:

$$
\pi=\left(\frac13,\frac13,\frac13\right)
$$

Matricea $A$ contine probabilitatile de tranzitie intre dificultati, iar matricea $B$ contine probabilitatile de emisie a fiecarei note pentru fiecare dificultate.

Verificarea sumelor liniilor confirma ca fiecare rand din $A$ si $B$ reprezinta o distributie de probabilitate.

In [1]:
import numpy as np

stari = ["D", "M", "U"]
note = ["FB", "B", "S", "NS"]

pi = np.array([
    1/3, 1/3, 1/3
])

A = np.array([
    [0.00, 0.50, 0.50],
    [0.50, 0.25, 0.25],
    [0.50, 0.25, 0.25]
])

B = np.array([
    [0.10, 0.20, 0.40, 0.30],  # D
    [0.15, 0.25, 0.50, 0.10],  # M
    [0.20, 0.30, 0.40, 0.10]   # U
])

print("Sume randuri A:", A.sum(axis=1))
print("Sume randuri B:", B.sum(axis=1))

Sume randuri A: [1. 1. 1.]
Sume randuri B: [1. 1. 1.]


In [2]:
observatii = [
    "FB", "FB", "S", "B", "B", "S",
    "B", "B", "NS", "B", "B", "S"
]

cod_note = {
    "FB": 0,
    "B": 1,
    "S": 2,
    "NS": 3
}

obs = [cod_note[x] for x in observatii]

### b. Initializarea Viterbi

Pentru prima observatie folosim formula:

$$
\delta_1(j)=\pi_jb_j(o_1)
$$

Prima nota observata este $FB$, deci:

$$
\delta_1(D)=\frac13\cdot0.10=0.03333
$$

$$
\delta_1(M)=\frac13\cdot0.15=0.05
$$

$$
\delta_1(U)=\frac13\cdot0.20=0.06667
$$

Prin urmare:

$$
\delta_1=(0.03333,\ 0.05,\ 0.06667)
$$

In [3]:
T = len(obs)
N = len(stari)

delta = np.zeros((T, N))
psi = np.zeros((T, N), dtype=int)

delta[0] = pi * B[:, obs[0]]

print("delta_1 =", delta[0])

delta_1 = [0.03333333 0.05       0.06666667]


In [4]:
for t in range(1, T):

    for j in range(N):

        scoruri = delta[t - 1] * A[:, j]

        psi[t, j] = np.argmax(scoruri)

        delta[t, j] = (
            B[j, obs[t]]
            * scoruri[psi[t, j]]
        )

### c. Recurenta Viterbi

Pentru fiecare moment $t$ si fiecare stare $j$ calculam:

$$
\delta_t(j)
=
b_j(o_t)
\max_i\left(\delta_{t-1}(i)a_{ij}\right)
$$

Tabelul $\delta$ pastreaza cel mai bun scor pentru fiecare stare la fiecare moment.

Tabelul $\psi$ pastreaza indicele starii precedente care a produs acel scor maxim:

$$
\psi_t(j)
=
\arg\max_i\left(\delta_{t-1}(i)a_{ij}\right)
$$

Valorile din $\psi$ vor fi folosite la final pentru reconstruirea traseului in ordine inversa.

In [5]:
print("delta_1 =", delta[0])
print("delta_2 =", delta[1])

delta_1 = [0.03333333 0.05       0.06666667]
delta_2 = [0.00333333 0.0025     0.00333333]


### d. Verificare intermediara

Dupa primele doua observatii $FB$, obtinem:

$$
\delta_1=(0.03333,\ 0.05,\ 0.06667)
$$

si:

$$
\delta_2=(0.003333,\ 0.0025,\ 0.003333)
$$

Verificam manual valoarea pentru starea $D$ la momentul 2:

$$
\delta_2(D)
=
0.1\max
\left(
0.03333\cdot0,\ 
0.05\cdot0.5,\ 
0.06667\cdot0.5
\right)
$$

$$
=
0.1\max(0,\ 0.025,\ 0.03333)
$$

$$
\boxed{\delta_2(D)=0.003333}
$$

Valoarea maxima provine din starea $U$, deci pentru acest caz $\psi$ retine starea $U$ ca stare precedenta.

In [6]:
ultima_stare = np.argmax(delta[T - 1])
probabilitate_max = delta[T - 1, ultima_stare]

In [7]:
traseu = np.zeros(T, dtype=int)

traseu[T - 1] = ultima_stare

In [8]:
for t in range(T - 1, 0, -1):
    traseu[t - 1] = psi[t, traseu[t]]

In [9]:
traseu_nume = [stari[i] for i in traseu]

print("Secventa observata:")
print(observatii)

print("\nSecventa Viterbi:")
print(traseu_nume)

print("\nProbabilitatea maxima:")
print(probabilitate_max)

Secventa observata:
['FB', 'FB', 'S', 'B', 'B', 'S', 'B', 'B', 'NS', 'B', 'B', 'S']

Secventa Viterbi:
['D', 'U', 'D', 'U', 'D', 'M', 'D', 'U', 'D', 'U', 'D', 'M']

Probabilitatea maxima:
2.1093750000000005e-11


### e. Reconstructia traseului

Dupa calcularea tuturor valorilor $\delta$, alegem starea finala cu scorul maxim. Apoi parcurgem tabelul $\psi$ in ordine inversa pentru a reconstrui intreaga secventa de dificultati.

Secventa obtinuta este:

$$
D,U,D,U,D,M,D,U,D,U,D,M
$$

iar scorul maxim este:

$$
\boxed{2.109375\cdot10^{-11}}
$$

### f. Interpretare

Algoritmul Viterbi cauta cea mai probabila **secventa completa** de dificultati, luand in considerare simultan probabilitatile de tranzitie si probabilitatile observatiilor.

De aceea, rezultatul nu trebuie confundat cu alegerea separata a celei mai probabile dificultati pentru fiecare test.

Valoarea obtinuta la final este probabilitatea comuna:

$$
P(x^*,o)
$$

adica probabilitatea traseului optim impreuna cu secventa de note observata. Ea nu reprezinta probabilitatea conditionata:

$$
P(x^*\mid o)
$$